# Longitudinal correlations of inStrain significant microbes with clinical severity
Date last updated: 1/1/2026

Notebook author: Yang Chen

In [24]:
# Import Python packages and plotting
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Ellipse
from matplotlib.ticker import MaxNLocator
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
from matplotlib.ticker import MaxNLocator

# Scientific and statistical packages
import scipy.stats as ss
from skbio import DistanceMatrix
from skbio.stats.ordination import OrdinationResults
from skbio.stats.distance import permanova

# BIOM and Qiime2 related packages
import biom
import qiime2 as q2
from biom import load_table

# Gemelli RPCA and rarefaction utilities
import gemelli
from gemelli.preprocessing import rclr_transformation
from gemelli.preprocessing import matrix_rclr

In [25]:
# Function to load BIOM table, sort rows by sum
def biom_to_df(biom_path):

    # Load BIOM table and convert to a DataFrame
    table = biom.load_table(biom_path)

    df = pd.DataFrame(table.matrix_data.toarray(),
                      index=table.ids(axis='observation'),
                      columns=table.ids(axis='sample'))
    
    
    # Sort rows by row sum in descending order
    df['row_sum'] = df.sum(axis=1)
    df = df.sort_values(by='row_sum', ascending=False)
    
    # Drop the 'row_sum' column before proceeding
    df = df.drop(columns=['row_sum'])

    # Transpose df
    df = df.T
    
    return df


In [26]:
V1V3_path = '../Data/16S/Tables/179426_feature-table_16S_V1V3_rare-11054_sampleIDfixed_Genus_SILVA.biom'
V1V3_df = biom_to_df(V1V3_path)
V1V3_df

,g__Cutibacterium,g__uncultured,g__Staphylococcus,g__Xanthomonas,g__Lawsonella,g__Streptococcus,g__Corynebacterium,g__Veillonella,g__Alloprevotella,g__Lactobacillus,...,g__Kocuria,g__Anaerococcus,g__Caulobacter,g__Nocardioides,g__Brevibacterium,g__Brochothrix,g__Faecalibacterium,g__Pseudoglutamicibacter,g__Brevundimonas,g__Pseudoalteromonas
LAMI.RD001.D0.C1,7219.0,2.0,177.0,0.0,241.0,852.0,478.0,89.0,46.0,3.0,...,0.0,0.0,0.0,2.0,4.0,0.0,0.0,10.0,2.0,60.0
LAMI.RD001.D0.C2,5374.0,3.0,466.0,0.0,391.0,1036.0,956.0,142.0,81.0,536.0,...,30.0,1.0,7.0,19.0,12.0,4.0,0.0,4.0,11.0,0.0
LAMI.RD001.D14.C1,6592.0,16.0,263.0,0.0,729.0,694.0,1613.0,82.0,95.0,193.0,...,0.0,4.0,8.0,1.0,45.0,5.0,0.0,12.0,0.0,0.0
LAMI.RD001.D28.C1,7066.0,12.0,360.0,0.0,601.0,932.0,1015.0,96.0,107.0,69.0,...,21.0,0.0,3.0,0.0,32.0,0.0,0.0,10.0,4.0,0.0
LAMI.RD002.D14.C1,10113.0,26.0,319.0,0.0,135.0,134.0,136.0,8.0,4.0,3.0,...,1.0,53.0,27.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD318.D28.C3,9420.0,1604.0,16.0,0.0,8.0,3.0,2.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD319.D14.C1,5440.0,5551.0,17.0,0.0,25.0,1.0,14.0,0.0,0.0,2.0,...,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD319.D14.C2,6887.0,4016.0,40.0,0.0,69.0,0.0,37.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD319.D9.C1,3551.0,7319.0,44.0,0.0,103.0,4.0,20.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [27]:
V4_path = '../Data/16S/Tables/174951_feature-table_16S_V4_rare-3769_sampleIDfixed_Genus_SILVA.biom'
V4_df = biom_to_df(V4_path)
V4_df

,g__Staphylococcus,g__uncultured,g__Lawsonella,g__Corynebacterium,g__Streptococcus,g__Acinetobacter,g__Haemophilus,g__Xanthomonas,g__Micrococcus,g__Lactobacillus,...,g__Anaerostipes,g__Treponema,g__Dietzia,g__Conchiformibius,g__Simonsiella,g__Hymenobacter,g__Bacteroides,g__Absconditabacteriales_(SR1),g__Sphingomonas,g__Pasteurella
LAMI.RD001.D0.C1,317.0,61.0,131.0,275.0,414.0,116.0,475.0,0.0,100.0,15.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,38.0,0.0,0.0
173620.14901.BLANK.LOREAL.2.2H,0.0,40.0,0.0,0.0,135.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD001.D14.C1,454.0,29.0,456.0,915.0,247.0,57.0,168.0,1.0,93.0,117.0,...,3.0,0.0,0.0,0.0,8.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD004.D0.C2,1652.0,36.0,95.0,848.0,164.0,72.0,28.0,0.0,81.0,97.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD001.D0.C2,349.0,46.0,217.0,509.0,368.0,83.0,464.0,0.0,75.0,182.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,19.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD319.D16.C2,142.0,3413.0,57.0,48.0,2.0,3.0,0.0,0.0,2.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD319.D28.C1,42.0,3525.0,96.0,38.0,7.0,5.0,9.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD318.D9.C2,139.0,3441.0,23.0,27.0,11.0,35.0,2.0,0.0,12.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD319.D28.C2,52.0,3627.0,54.0,15.0,1.0,0.0,1.0,1.0,2.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [28]:
wgs_path = '../Data/metaG/Tables/VEBA_MAGs_collapsed_genus_absolute.biom'
# wgs_path = '../Data/metaG/Tables/metaG_rs210_micov-filt_genus.biom'
wgs_df = biom_to_df(wgs_path).T
wgs_df.index = wgs_df.index.str.replace('_', '.', regex=False)
wgs_df.index = wgs_df.index.to_series().str.split('.').str[:4].str.join('.')
wgs_df.index = wgs_df.index.str.replace(' g', 'g', regex=False)

# wgs_df = wgs_df.rename(columns={'Corynebacterium spp.': ' g__Corynebacterium'})
# wgs_df = wgs_df.rename(columns={'Micrococcus spp.': ' g__Micrococcus'})
# wgs_df = wgs_df.rename(columns={'Lawsonella cleavelandensis': ' g__Lawsonella'})
# wgs_df = wgs_df.rename(columns={'Neisseria spp.': ' g__Neisseria'})
# wgs_df[' g__Cutibacterium'] = wgs_df[' g__Cutibacterium acnes'] + wgs_df[' g__Cutibacterium granulosum']

wgs_df

,g__,g__Abiotrophia,g__Acidovorax,g__Acinetobacter,g__Actinomyces,g__Aerococcus,g__Afipia,g__Aggregatibacter,g__Alloiococcus,g__Alloprevotella,...,g__Stomatobaculum,g__Streptococcus,g__Tannerella,g__Tepidiphilus,g__Tsuneonella,g__Varibaculum,g__Veillonella,g__Winkia,g__Xanthomonas,g__Yonghaparkia
LAMI.RD304.D0.C1,0.000000,0.000012,0.000082,0.002366,0.001358,0.000070,0.000047,0.000012,0.000000,0.000000,...,0.000012,0.000304,0.000000,0.000012,0.000023,0.000187,0.000059,0.000023,0.000023,0.000023
LAMI.RD310.D14.C3,0.000000,0.000000,0.000000,0.000000,0.000117,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000094,0.000000,0.000000,0.000000,0.000000,0.000012,0.000000,0.000000,0.000012
LAMI.RD310.D0.C3,0.000000,0.000000,0.000000,0.000047,0.000445,0.000012,0.000035,0.000000,0.000000,0.000000,...,0.000000,0.000141,0.000000,0.000152,0.000000,0.000000,0.000012,0.000000,0.000094,0.000000
LAMI.RD304.D14.C3,0.000000,0.000082,0.000000,0.000094,0.006804,0.000023,0.000000,0.000117,0.000000,0.000422,...,0.000000,0.004942,0.000164,0.000000,0.000000,0.000012,0.000632,0.000047,0.000012,0.000000
LAMI.RD306.D11.C1,0.000035,0.000527,0.000047,0.000539,0.010645,0.000000,0.000047,0.000023,0.000070,0.000480,...,0.000012,0.002611,0.000012,0.000000,0.000023,0.000023,0.000504,0.000000,0.000082,0.000035
LAMI.RD304.D28.C3,0.000000,0.000000,0.000023,0.000000,0.000609,0.000012,0.000012,0.000000,0.000000,0.000012,...,0.000000,0.000129,0.000000,0.000000,0.000000,0.000105,0.000023,0.000035,0.000000,0.000000
LAMI.RD310.D7.C3,0.000000,0.000000,0.000000,0.000000,0.000164,0.000000,0.000012,0.000000,0.000000,0.000000,...,0.000000,0.000059,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
LAMI.RD308.D0.C3,0.000000,0.000000,0.000000,0.000023,0.000913,0.000000,0.000000,0.000000,0.000000,0.000012,...,0.000000,0.000117,0.000000,0.000000,0.000000,0.000012,0.000012,0.000012,0.000012,0.000000
LAMI.RD308.D11.C2,0.000012,0.000000,0.000059,0.000785,0.001429,0.000000,0.000012,0.000000,0.000012,0.000152,...,0.000000,0.000550,0.000000,0.000000,0.000059,0.000047,0.000012,0.000023,0.000023,0.000000
LAMI.RD306.D14.C3,0.000023,0.000211,0.000035,0.000363,0.010434,0.000000,0.000047,0.000012,0.000059,0.000281,...,0.000012,0.002389,0.000012,0.000000,0.000035,0.000012,0.000164,0.000000,0.000012,0.000035


In [29]:
# wgs metadata
wgs_metadata = pd.read_csv('../Metadata/wgs_simplified_md.tsv', sep='\t')
wgs_individuals = wgs_metadata['sample_name'].str.extract(r'(RD\d{3})')[0].tolist()

In [30]:
# Import in the metadata
metadata = pd.read_csv('../Metadata/metadata_final_22102024.tsv', sep='\t')
# metadata = pd.read_csv('../Metadata/wgs_simplified_md.tsv', sep='\t')
metadata.columns
metadata = metadata[['SampleID', 'local_lesion_severity', 'host_subject_id', 'day']]
metadata = metadata.set_index('SampleID')

# metadata = metadata[['sample_name', 'local_lesion_severity', 'subject_ID', 'day']]
# metadata = metadata.set_index('sample_name')
metadata.index.name = None

indexes_to_include = V4_df.index.to_list()

metadata = metadata.loc[metadata.index.isin(indexes_to_include)]
metadata

,local_lesion_severity,host_subject_id,day
LAMI.RD308.D16.C1,4,RD308,16
LAMI.RD310.D21.C1,4,RD310,21
LAMI.RD305.D21.C3,0,RD305,21
LAMI.RD306.D18.C2,2,RD306,18
LAMI.RD306.D7.C2,3,RD306,7
...,...,...,...
LAMI.RD305.D0.C2,3,RD305,0
LAMI.RD317.D21.C1,1,RD317,21
LAMI.RD001.D0.C1,0,RD001,0
LAMI.RD314.D0.C1,1,RD314,0


In [31]:
all_individuals = metadata.index.to_series().str.split('.').str[1].tolist()
# Extract from index and remove 'RD' prefix
all_individuals_numeric_only = metadata.index.to_series().str.split('.').str[1].str.replace('RD', '', regex=False).tolist()

## Plot correlation figures

In [32]:
# for participant_ID in wgs_individuals:

#     md_pp = metadata[metadata['host_subject_id'] == participant_ID]
#     md_pp = md_pp[md_pp.index.str.endswith(zone)]
#     md_pp_samples = md_pp.index.to_list()

#     # Subset data tables to be same samples
#     V1V3_df_md_pp = V1V3_df.loc[V1V3_df.index.isin(md_pp_samples)]
#     V4_df_md_pp = V4_df.loc[V4_df.index.isin(md_pp_samples)]
#     # wgs_df_md_pp = wgs_df.loc[wgs_df.index.isin(md_pp_samples)]

#     # Add pseudocount
#     V1V3_df_md_pp_pseudo = V1V3_df_md_pp + 1
#     V4_df_md_pp_pseudo = V4_df_md_pp + 1
#     # wgs_df_md_pp_pseudo = wgs_df_md_pp + 1

#     # CLR transformation function
#     def clr_transform(df):
#         log_df = np.log(df)
#         geometric_mean = log_df.mean(axis=1)
#         clr_df = log_df.subtract(geometric_mean, axis=0)
#         return clr_df

#     # Apply CLR
#     V1V3_df_md_pp_clr = clr_transform(V1V3_df_md_pp_pseudo)
#     V4_df_md_pp_clr = clr_transform(V4_df_md_pp_pseudo)
#     # wgs_df_md_pp_clr = clr_transform(wgs_df_md_pp_pseudo)   

#     md_pp[f'{taxa} V1V3'] = V1V3_df_md_pp_clr[f' {taxa}']
#     md_pp[f'{taxa} V4'] = V4_df_md_pp_clr[f' {taxa}']
#     # md_pp[f'{taxa} WGS'] = wgs_df_md_pp_clr[f' {taxa}']

#     md_pp = md_pp.dropna(subset=[f'{taxa} V1V3', f'{taxa} V4'])


#     # Sort by day for better plotting
#     df = md_pp.sort_values('day')

#     # df_wgs = df.dropna(subset=[f'{taxa} WGS', 'day'])

#     # Correlation: V1V3
#     r1, p1 = spearmanr(df['local_lesion_severity'], df[f'{taxa} V1V3'])

#     # Correlation: V4
#     r2, p2 = spearmanr(df['local_lesion_severity'], df[f'{taxa} V4'])

#     # Correlation: WGS — use dropna to avoid errors
#     # df_clean = df_wgs[['local_lesion_severity', f'{taxa} WGS']].dropna()
#     # r3, p3 = spearmanr(df_clean['local_lesion_severity'], df_clean[f'{taxa} WGS'])

#     # Start figure
#     fig, ax1 = plt.subplots(figsize=(6, 6))

#     # Plot local lesion severity
#     ax1.plot(df['day'], df['local_lesion_severity'], color='red', marker='o', label='Local Lesion Severity')
    
#     if participant_ID == 'RD304' and zone == 'C1':
#         ax1.set_ylabel(f'Local Lesion Severity: Participant 1_L1', color='red', fontsize=12)
#     elif participant_ID == 'RD306' and zone == 'C1':
#         ax1.set_ylabel(f'Local Lesion Severity: Participant 2_L1', color='red', fontsize=12)
#     elif participant_ID == 'RD308' and zone == 'C1':
#         ax1.set_ylabel(f'Local Lesion Severity: Participant 3_L1', color='red', fontsize=12)
#     elif participant_ID == 'RD310' and zone == 'C1':
#         ax1.set_ylabel(f'Local Lesion Severity: Participant 4_L1', color='red', fontsize=12)


#     ax1.tick_params(axis='y', labelcolor='black')

#     if taxa == 'g__Cutibacterium':
#         ax1.set_ylim(-1, 10)
#     else:     
#         ax1.set_ylim(-1, 6)

#     # Secondary y-axis for taxa CLR
#     ax2 = ax1.twinx()

#     # V1V3 (light blue squares)
#     ax2.plot(df['day'], df[f'{taxa} V1V3'], color='#86ceeb', marker='s', label=f'{taxa} (V1V3)')

#     # V4 (purple diamonds)
#     ax2.plot(df['day'], df[f'{taxa} V4'], color='#dda0dd', marker='D', label=f'{taxa} (V4)')

#     # WGS (orange triangles)
#     # ax2.plot(
#     #     df_wgs['day'], 
#     #     df_wgs[f'{taxa} WGS'], 
#     #     color='orange', marker='^', label=f'{taxa} (WGS)'
#     # )

#     ax2.set_ylabel(f'{taxa} Abundance (CLR)', color='black', fontsize=12)
#     ax2.tick_params(axis='y', labelcolor='black')
    
#     if taxa == 'g__Cutibacterium':
#         ax2.set_ylim(-1, 10)
#     else:     
#         ax2.set_ylim(-1, 6)

#     # Title and labels
#     # plt.title("Subject 304: Acne Severity vs. Corynebacterium Abundance", fontsize=13)
#     ax1.set_xlabel('Day', color='black', fontsize=12)

#     # Annotate V4 first (higher up)
#     ax1.text(0.98, 0.95, f"V4: Spearman (ρ)={r2:.2f}, p={'< 0.001' if p2 < 0.001 else f'{p2:.3f}'}",
#             transform=ax1.transAxes, fontsize=10, ha='right')

#     # Annotate V1V3 second (under V4)
#     ax1.text(0.98, 0.91, f"V1V3: Spearman (ρ)={r1:.2f}, p={'< 0.001' if p1 < 0.001 else f'{p1:.3f}'}",
#             transform=ax1.transAxes, fontsize=10, ha='right')

#     # ax1.text(0.4, 0.87, f"WGS: Spearman (ρ)={r3:.2f}, p={'< 0.001' if p3 < 0.001 else f'{p3:.3f}'}", transform=ax1.transAxes, fontsize=10)

#     # Combine legends from both axes
#     lines1, labels1 = ax1.get_legend_handles_labels()
#     lines2, labels2 = ax2.get_legend_handles_labels()
#     # lines3, labels3 = ax2.get_legend_handles_labels()

#     ax1.legend(lines1 + lines2, labels1 + labels2, 
#             loc='upper center', bbox_to_anchor=(0.5, -0.15), 
#             ncol=2, fontsize=10, frameon=True)

#     # Tidy layout and save
#     plt.tight_layout()
#     plt.savefig(f'../Figures/multi-omics_Figures/individual_severity_correlations/{taxa}/{participant_ID}_{zone}_{taxa}_V1V3_V4.png', dpi=600)
#     plt.savefig(f'../Figures/multi-omics_Figures/individual_severity_correlations/{taxa}/{participant_ID}_{zone}_{taxa}_V1V3_V4.svg')


In [33]:
# Define taxa to analyze
taxa_list = ['g__Cutibacterium', 'g__Corynebacterium', 'g__Lawsonella']
zone = 'C1'

for taxa in taxa_list:
    print(f"\nProcessing {taxa}...")
    
    # Store all p-values for correction
    p_values_v1v3 = []
    p_values_v4 = []
    correlations_v1v3 = []
    correlations_v4 = []
    participant_data = []

    for participant_ID in wgs_individuals:

        md_pp = metadata[metadata['host_subject_id'] == participant_ID]
        md_pp = md_pp[md_pp.index.str.endswith(zone)]
        md_pp_samples = md_pp.index.to_list()

        # Subset data tables to be same samples
        V1V3_df_md_pp = V1V3_df.loc[V1V3_df.index.isin(md_pp_samples)]
        V4_df_md_pp = V4_df.loc[V4_df.index.isin(md_pp_samples)]
        # wgs_df_md_pp = wgs_df.loc[wgs_df.index.isin(md_pp_samples)]

        # Apply rCLR transformation using matrix_rclr (works directly with numpy/pandas)
        # matrix_rclr expects samples as rows, features as columns
        V1V3_df_md_pp_rclr = matrix_rclr(V1V3_df_md_pp.values)
        V1V3_df_md_pp_rclr = pd.DataFrame(V1V3_df_md_pp_rclr, 
                                           index=V1V3_df_md_pp.index, 
                                           columns=V1V3_df_md_pp.columns)
        
        V4_df_md_pp_rclr = matrix_rclr(V4_df_md_pp.values)
        V4_df_md_pp_rclr = pd.DataFrame(V4_df_md_pp_rclr, 
                                         index=V4_df_md_pp.index, 
                                         columns=V4_df_md_pp.columns)
        
        # wgs_df_md_pp_rclr = matrix_rclr(wgs_df_md_pp.values)
        # wgs_df_md_pp_rclr = pd.DataFrame(wgs_df_md_pp_rclr, 
        #                                   index=wgs_df_md_pp.index, 
        #                                   columns=wgs_df_md_pp.columns)

        md_pp[f'{taxa} V1V3'] = V1V3_df_md_pp_rclr[f'{taxa}']
        md_pp[f'{taxa} V4'] = V4_df_md_pp_rclr[f'{taxa}']
        # md_pp[f'{taxa} WGS'] = wgs_df_md_pp_rclr[f' {taxa}']

        md_pp = md_pp.dropna(subset=[f'{taxa} V1V3', f'{taxa} V4'])

        # Sort by day for better plotting
        df = md_pp.sort_values('day')

        # df_wgs = df.dropna(subset=[f'{taxa} WGS', 'day'])

        # Correlation: V1V3
        r1, p1 = spearmanr(df['local_lesion_severity'], df[f'{taxa} V1V3'])

        # Correlation: V4
        r2, p2 = spearmanr(df['local_lesion_severity'], df[f'{taxa} V4'])

        # Correlation: WGS — use dropna to avoid errors
        # df_clean = df_wgs[['local_lesion_severity', f'{taxa} WGS']].dropna()
        # r3, p3 = spearmanr(df_clean['local_lesion_severity'], df_clean[f'{taxa} WGS'])

        # Store p-values and correlations for later correction
        p_values_v1v3.append(p1)
        p_values_v4.append(p2)
        correlations_v1v3.append(r1)
        correlations_v4.append(r2)
        participant_data.append((participant_ID, df))

    # Apply Benjamini-Hochberg correction
    # Combine all p-values for correction across both methods
    all_p_values = p_values_v1v3 + p_values_v4
    reject, corrected_p_values, _, _ = multipletests(all_p_values, method='fdr_bh')

    # Split corrected p-values back
    n_participants = len(wgs_individuals)
    corrected_p_v1v3 = corrected_p_values[:n_participants]
    corrected_p_v4 = corrected_p_values[n_participants:]

    # Now create plots with corrected p-values
    for idx, (participant_ID, df) in enumerate(participant_data):
        
        r1 = correlations_v1v3[idx]
        r2 = correlations_v4[idx]
        p1_corrected = corrected_p_v1v3[idx]
        p2_corrected = corrected_p_v4[idx]

        # Start figure
        fig, ax1 = plt.subplots(figsize=(6, 6))

        # Plot local lesion severity
        ax1.plot(df['day'], df['local_lesion_severity'], color='red', marker='o', label='Local Lesion Severity')
        
        if participant_ID == 'RD304' and zone == 'C1':
            ax1.set_ylabel(f'Local Lesion Severity: Participant 1_L1', color='red', fontsize=12)
        elif participant_ID == 'RD306' and zone == 'C1':
            ax1.set_ylabel(f'Local Lesion Severity: Participant 2_L1', color='red', fontsize=12)
        elif participant_ID == 'RD308' and zone == 'C1':
            ax1.set_ylabel(f'Local Lesion Severity: Participant 3_L1', color='red', fontsize=12)
        elif participant_ID == 'RD310' and zone == 'C1':
            ax1.set_ylabel(f'Local Lesion Severity: Participant 4_L1', color='red', fontsize=12)

        # Force integer ticks on primary y-axis (lesion severity)
        ax1.yaxis.set_major_locator(MaxNLocator(integer=True))
        ax1.tick_params(axis='y', labelcolor='black')

        # Set y-axis limits based on taxa
        if taxa == 'g__Cutibacterium':
            ax1.set_ylim(-3, 10)
        elif taxa == 'g__Lawsonella':
            ax1.set_ylim(-3, 6)
        elif taxa == 'g__Corynebacterium':
            ax1.set_ylim(-2, 6)

        # Secondary y-axis for taxa rCLR
        ax2 = ax1.twinx()

        # V1V3 (light blue squares)
        ax2.plot(df['day'], df[f'{taxa} V1V3'], color='#86ceeb', marker='s', label=f'{taxa} (V1V3)')

        # V4 (purple diamonds)
        ax2.plot(df['day'], df[f'{taxa} V4'], color='#dda0dd', marker='D', label=f'{taxa} (V4)')

        # WGS (orange triangles)
        # ax2.plot(
        #     df_wgs['day'], 
        #     df_wgs[f'{taxa} WGS'], 
        #     color='orange', marker='^', label=f'{taxa} (WGS)'
        # )

        ax2.set_ylabel(f'{taxa} Abundance (rCLR)', color='black', fontsize=12)
        
        # Set secondary y-axis to same limits as primary y-axis
        if taxa == 'g__Cutibacterium':
            ax2.set_ylim(-3, 10)
        elif taxa == 'g__Lawsonella':
            ax2.set_ylim(-3, 6)
        elif taxa == 'g__Corynebacterium':
            ax2.set_ylim(-2, 6)
        
        # Force integer ticks on secondary y-axis
        ax2.yaxis.set_major_locator(MaxNLocator(integer=True))
        ax2.tick_params(axis='y', labelcolor='black')

        # Title and labels
        # plt.title("Subject 304: Acne Severity vs. Corynebacterium Abundance", fontsize=13)
        ax1.set_xlabel('Day', color='black', fontsize=12)

        # Format corrected p-values properly
        p1_text = '< 0.001' if p1_corrected < 0.001 else f'= {p1_corrected:.3f}'
        p2_text = '< 0.001' if p2_corrected < 0.001 else f'= {p2_corrected:.3f}'

        # Annotate V4 first (higher up) with FDR-corrected p-values
        ax1.text(0.98, 0.95, f"V4: Spearman ρ = {r2:.2f}, FDR-corrected p {p2_text}",
                transform=ax1.transAxes, fontsize=10, ha='right')

        # Annotate V1V3 second (under V4)
        ax1.text(0.98, 0.91, f"V1V3: Spearman ρ = {r1:.2f}, FDR-corrected p {p1_text}",
                transform=ax1.transAxes, fontsize=10, ha='right')

        # p3_text = '< 0.001' if p3_corrected < 0.001 else f'= {p3_corrected:.3f}'
        # ax1.text(0.4, 0.87, f"WGS: Spearman ρ = {r3:.2f}, FDR-corrected p {p3_text}", transform=ax1.transAxes, fontsize=10)

        # Combine legends from both axes
        lines1, labels1 = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        # lines3, labels3 = ax2.get_legend_handles_labels()

        ax1.legend(lines1 + lines2, labels1 + labels2, 
                loc='upper center', bbox_to_anchor=(0.5, -0.15), 
                ncol=2, fontsize=10, frameon=True)

        # Tidy layout and save
        plt.tight_layout()
        if participant_ID == 'RD304':
            plt.savefig(f'../Figures/Main/Figure_3/Severity_corr-{participant_ID}_{zone}_{taxa}.png', dpi=600)
        else:
            plt.savefig(f'../Figures/Supplementary/Suppl_Figure_6/Severity_corr-{participant_ID}_{zone}_{taxa}.png', dpi=600)
        plt.close()
    
    print(f"Completed {taxa}")


Processing g__Cutibacterium...
Completed g__Cutibacterium

Processing g__Corynebacterium...
Completed g__Corynebacterium

Processing g__Lawsonella...
Completed g__Lawsonella
